In [10]:
import numpy as np
import os 
import librosa

In [8]:
dataset_path =r"C:\Users\KHALIL\Desktop\reconnaissance-vocal\Dataset"

X = []
y = []


In [ ]:
# extraire les features avec MFCC
for person in os.listdir(dataset_path):
    person_folder = os.path.join(dataset_path, person)
     
    for file_name in os.listdir(person_folder):
                file_path = os.path.join(person_folder, file_name)
                
                audio, sr = librosa.load(file_path, sr=None)
                mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=13)
                mfcc_mean = np.mean(mfcc, axis=1)
                
                X.append(mfcc_mean)
                y.append(person)

In [12]:
X = np.array(X)
y = np.array(y)

print("Shape de X :", X.shape)
print("Shape de y :", y.shape)

Shape de X : (55, 13)
Shape de y : (55,)


In [ ]:
#Encodage des classes
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y)

print("Classes :", encoder.classes_)
print("y encodé :", y_encoded)

Classes : ['Farissi_Fatimaezzahra' 'Fatima ezzahra talhi' 'Halima_Moutcho'
 'Hiba_ELIdrisy' 'Imane_Chalati' 'Khadija_El_boudhiri'
 'Oueld Aadou Kawtar' 'Oumaima_Khalil' 'Salma_Nghira' 'Sara Domti'
 'Zaitoun Dounia']
y encodé : [ 0  0  0  0  0  1  1  1  1  1  2  2  2  2  2  3  3  3  3  3  4  4  4  4
  4  5  5  5  5  5  6  6  6  6  6  7  7  7  7  7  8  8  8  8  8  9  9  9
  9  9 10 10 10 10 10]


In [ ]:
# Separation en test et train
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42
)

print("X_train :", X_train.shape)
print("X_test :", X_test.shape)
print("y_train :", y_train.shape)
print("y_test :", y_test.shape)

X_train : (44, 13)
X_test : (11, 13)
y_train : (44,)
y_test : (11,)


In [17]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [18]:
# Nous avons choisi un modèle Dense car nos données sont déjà sous forme de vecteurs de caractéristiques (MFCC)
#  ce qui rend l’utilisation d’un CNN inutile

model = Sequential()

model.add(Dense(64, activation='relu', input_shape=(13,)))
model.add(Dense(32, activation='relu'))
model.add(Dense(len(encoder.classes_), activation='softmax'))

c:\Users\KHALIL\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [20]:
# copiler le model 

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [21]:
#Entrainement du model

history = model.fit(
    X_train, y_train,
    epochs=30,
    validation_data=(X_test, y_test)
)

Epoch 1/30
2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 334ms/step - accuracy: 0.0909 - loss: 73.3244 - val_accuracy: 0.1818 - val_loss: 54.2646
Epoch 2/30
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step - accuracy: 0.1591 - loss: 62.0420 - val_accuracy: 0.1818 - val_loss: 49.5260
Epoch 3/30
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step - accuracy: 0.1364 - loss: 53.8563 - val_accuracy: 0.1818 - val_loss: 45.1561
Epoch 4/30
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step - accuracy: 0.0455 - loss: 48.4434 - val_accuracy: 0.1818 - val_loss: 38.7984
Epoch 5/30
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step - accuracy: 0.0227 - loss: 42.4377 - val_accuracy: 0.0909 - val_loss: 34.5459
Epoch 6/30
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step - accuracy: 0.0455 - loss: 36.2883 - val_accuracy: 0.1818 - val_loss: 29.5879
Epoch 7/30
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step - accuracy: 0.1136 - loss: 31.0789 - val_accuracy: 0.0909 - val_loss: 26.1997
Epoch 8/30
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 108ms/step - accuracy: 0.1591 - loss: 27.8475 - val_accuracy: 0.0909 - 

In [22]:
import sounddevice as sd
from scipy.io.wavfile import write
import librosa
import numpy as np

def reconnaitre_par_micro(model, encoder, duration=4, fs=48000, output_file="test_micro.wav"):
    print("Parle maintenant...")

    recording = sd.rec(int(duration * fs), samplerate=fs, channels=1, dtype='float32')
    sd.wait()

    print("Enregistrement terminé.")

    write(output_file, fs, recording)

    audio, sr = librosa.load(output_file, sr=None)
    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=13)
    mfcc_mean = np.mean(mfcc, axis=1)
    mfcc_mean = mfcc_mean.reshape(1, -1)

    prediction = model.predict(mfcc_mean)
    predicted_class = np.argmax(prediction)
    predicted_name = encoder.inverse_transform([predicted_class])[0]
    confidence = np.max(prediction)

    print("Personne reconnue :", predicted_name)
    print("Confiance :", confidence)

    return predicted_name, confidence

In [27]:
reconnaitre_par_micro(model, encoder)

Parle maintenant...
Enregistrement terminé.
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
Personne reconnue : Hiba_ELIdrisy
Confiance : 0.9676695


(np.str_('Hiba_ELIdrisy'), np.float32(0.9676695))